In [0]:
# CELL 1: Fetch credentials and configure Snowflake connection
snowflake_url = dbutils.secrets.get(scope="retail-scope", key="snowflake-url")
snowflake_user = dbutils.secrets.get(scope="retail-scope", key="snowflake-user")
snowflake_password = dbutils.secrets.get(scope="retail-scope", key="snowflake-password")

sfOptions = {
    "sfURL": snowflake_url,
    "sfUser": snowflake_user,
    "sfPassword": snowflake_password,
    "sfDatabase": "RETAIL_CAPSTONE_DB",
    "sfSchema": "GOLD",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

print("✅ Snowflake connection configured for GOLD schema!")


In [0]:
# CELL 2: Rebuild all 5 Gold tables directly in Snowflake
gold_models = [
    ("GOLD.DIM_DATE", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.DIM_DATE AS
        WITH distinct_dates AS (
            SELECT DISTINCT DATE AS sale_date
            FROM RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN
            WHERE DATE IS NOT NULL
        )
        SELECT
            sale_date AS date_id,
            sale_date AS full_date,
            EXTRACT(YEAR FROM sale_date)                                  AS year,
            EXTRACT(QUARTER FROM sale_date)                               AS quarter,
            EXTRACT(MONTH FROM sale_date)                                 AS month,
            TO_CHAR(sale_date, 'Mon')                                     AS month_name,
            EXTRACT(WEEK FROM sale_date)                                  AS week_of_year,
            EXTRACT(DAY FROM sale_date)                                   AS day_of_month,
            DAYOFWEEK(sale_date)                                          AS day_of_week,
            DAYNAME(sale_date)                                            AS day_name,
            CASE WHEN DAYOFWEEK(sale_date) IN (0, 6) THEN 1 ELSE 0 END   AS is_weekend,
            CASE WHEN EXTRACT(MONTH FROM sale_date) IN (11, 12) THEN 1 ELSE 0 END AS is_holiday_season
        FROM distinct_dates
    """),

    ("GOLD.DIM_STORE", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.DIM_STORE AS
        SELECT
            STORE AS store_id,
            STORETYPE AS store_type,
            CASE 
                WHEN STORETYPE = 'a' THEN 'Type A (Standard)'
                WHEN STORETYPE = 'b' THEN 'Type B (High-Volume / Extra)'
                WHEN STORETYPE = 'c' THEN 'Type C (Medium)'
                ELSE 'Type D (Extended)'
            END AS store_type_name,
            ASSORTMENT AS assortment,
            CASE 
                WHEN ASSORTMENT = 'a' THEN 'Basic'
                WHEN ASSORTMENT = 'b' THEN 'Extra'
                ELSE 'Extended'
            END AS assortment_name,
            COMPETITIONDISTANCE AS competition_distance,
            CASE
                WHEN COMPETITIONDISTANCE < 500   THEN '<500m'
                WHEN COMPETITIONDISTANCE <= 2000 THEN '500m-2km'
                WHEN COMPETITIONDISTANCE <= 5000 THEN '2km-5km'
                ELSE '>5km'
            END AS competition_distance_tier,
            PROMO2 AS is_promo2,
            PROMO2SINCEWEEK AS promo2_since_week,
            PROMO2SINCEYEAR AS promo2_since_year,
            PROMOINTERVAL AS promo_interval
        FROM RETAIL_CAPSTONE_DB.RAW.STORE_CLEANED
    """),

    ("GOLD.FACT_SALES", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.FACT_SALES AS
        SELECT
            MD5(CONCAT(CAST(STORE AS VARCHAR), '_', CAST(DATE AS VARCHAR))) AS sales_fact_key,
            STORE                                                           AS store_id,
            DATE                                                            AS sale_date,
            DAYOFWEEK                                                       AS day_of_week,
            SALES                                                           AS sales_amount,
            CUSTOMERS                                                       AS customer_count,
            ROUND(SALES / NULLIF(CUSTOMERS, 0), 2)                          AS sales_per_customer,
            PROMO                                                           AS is_promo,
            PROMO                                                           AS is_promo_active,
            STATEHOLIDAY                                                    AS state_holiday_code,
            CASE WHEN STATEHOLIDAY != '0' THEN 1 ELSE 0 END                 AS is_state_holiday,
            SCHOOLHOLIDAY                                                   AS is_school_holiday,
            CASE WHEN DAYOFWEEK IN (6, 7) THEN 1 ELSE 0 END                 AS is_weekend
        FROM RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN
        WHERE OPEN = 1 AND SALES > 0
    """),

    ("GOLD.FACT_STORE_CLOSURES", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.FACT_STORE_CLOSURES AS
        SELECT
            MD5(CONCAT(CAST(STORE AS VARCHAR), '_', CAST(DATE AS VARCHAR))) AS closure_fact_key,
            STORE                                                           AS store_id,
            DATE                                                            AS sale_date,
            DAYOFWEEK                                                       AS day_of_week,
            SALES                                                           AS sales_amount,
            CUSTOMERS                                                       AS customer_count,
            OPEN                                                            AS is_open,
            STATEHOLIDAY                                                    AS state_holiday_code,
            SCHOOLHOLIDAY                                                   AS is_school_holiday,
            CASE 
                WHEN STATEHOLIDAY IN ('a', 'b', 'c') THEN 'State Holiday Closure'
                WHEN SCHOOLHOLIDAY = 1               THEN 'School Holiday Closure'
                WHEN DAYOFWEEK = 7                   THEN 'Sunday / Weekend Closure'
                ELSE 'Operational / Refurbishment Closure'
            END AS closure_reason
        FROM RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN
        WHERE OPEN = 0 OR SALES = 0
    """),

    ("GOLD.AGG_STORE_PERFORMANCE", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.AGG_STORE_PERFORMANCE AS
        SELECT
            s.store_id,
            st.store_type_name,
            st.assortment_name,
            COUNT(s.sale_date)                             AS total_open_days,
            ROUND(SUM(s.sales_amount), 2)                  AS total_revenue,
            ROUND(AVG(s.sales_amount), 2)                  AS avg_daily_sales,
            SUM(s.customer_count)                          AS total_customers,
            ROUND(AVG(s.customer_count), 1)                AS avg_daily_customers,
            ROUND(AVG(s.sales_per_customer), 2)            AS avg_basket_size,
            ROUND(AVG(CASE WHEN s.is_promo = 1 THEN s.sales_amount END), 2) AS avg_promo_sales,
            ROUND(AVG(CASE WHEN s.is_promo = 0 THEN s.sales_amount END), 2) AS avg_non_promo_sales,
            ROUND(
                (AVG(CASE WHEN s.is_promo = 1 THEN s.sales_amount END) - 
                 AVG(CASE WHEN s.is_promo = 0 THEN s.sales_amount END)) / 
                NULLIF(AVG(CASE WHEN s.is_promo = 0 THEN s.sales_amount END), 0) * 100, 2
            ) AS promo_lift_pct
        FROM RETAIL_CAPSTONE_DB.GOLD.FACT_SALES s
        JOIN RETAIL_CAPSTONE_DB.GOLD.DIM_STORE st ON s.store_id = st.store_id
        GROUP BY s.store_id, st.store_type_name, st.assortment_name
    """)
]

# Execute each model synchronously via Snowflake native preactions
for model_name, query_sql in gold_models:
    print(f"Rebuilding {model_name}...")
    clean_sql = query_sql.strip().rstrip(";")
    spark.read.format("snowflake") \
        .options(**sfOptions) \
        .option("preactions", clean_sql) \
        .option("query", "SELECT 1 AS DUMMY") \
        .load() \
        .collect()
    print(f"  ✅ {model_name} successfully rebuilt in Snowflake!")

print("\n🎉 Success! All 5 Gold tables are 100% updated in Snowflake.")


In [0]:
# Databricks notebook source
# ============================================================
# TASK 4: Automated Snowflake Gold Layer Pipeline
# Rebuilds all 5 Gold tables directly in Snowflake from RAW
# ============================================================

# 1. Load credentials from retail-scope
snowflake_url = dbutils.secrets.get(scope="retail-scope", key="snowflake-url")
snowflake_user = dbutils.secrets.get(scope="retail-scope", key="snowflake-user")
snowflake_password = dbutils.secrets.get(scope="retail-scope", key="snowflake-password")

# 2. Snowflake connection options
sfOptions = {
    "sfURL": snowflake_url,
    "sfUser": snowflake_user,
    "sfPassword": snowflake_password,
    "sfDatabase": "RETAIL_CAPSTONE_DB",
    "sfSchema": "GOLD",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

print("✅ Connected to Snowflake. Starting automated Gold model build...")

# 3. The 5 Gold Model SQL Transformations
gold_models = [
    ("GOLD.DIM_DATE", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.DIM_DATE AS
        WITH distinct_dates AS (
            SELECT DISTINCT DATE AS sale_date
            FROM RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN
            WHERE DATE IS NOT NULL
        )
        SELECT
            sale_date AS date_id,
            sale_date AS full_date,
            EXTRACT(YEAR FROM sale_date)                                  AS year,
            EXTRACT(QUARTER FROM sale_date)                               AS quarter,
            EXTRACT(MONTH FROM sale_date)                                 AS month,
            TO_CHAR(sale_date, 'Mon')                                     AS month_name,
            EXTRACT(WEEK FROM sale_date)                                  AS week_of_year,
            EXTRACT(DAY FROM sale_date)                                   AS day_of_month,
            DAYOFWEEK(sale_date)                                          AS day_of_week,
            DAYNAME(sale_date)                                            AS day_name,
            CASE WHEN DAYOFWEEK(sale_date) IN (0, 6) THEN 1 ELSE 0 END   AS is_weekend,
            CASE WHEN EXTRACT(MONTH FROM sale_date) IN (11, 12) THEN 1 ELSE 0 END AS is_holiday_season
        FROM distinct_dates
    """),

    ("GOLD.DIM_STORE", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.DIM_STORE AS
        SELECT
            STORE AS store_id,
            STORETYPE AS store_type,
            CASE 
                WHEN STORETYPE = 'a' THEN 'Type A (Standard)'
                WHEN STORETYPE = 'b' THEN 'Type B (High-Volume / Extra)'
                WHEN STORETYPE = 'c' THEN 'Type C (Medium)'
                ELSE 'Type D (Extended)'
            END AS store_type_name,
            ASSORTMENT AS assortment,
            CASE 
                WHEN ASSORTMENT = 'a' THEN 'Basic'
                WHEN ASSORTMENT = 'b' THEN 'Extra'
                ELSE 'Extended'
            END AS assortment_name,
            COMPETITIONDISTANCE AS competition_distance,
            CASE
                WHEN COMPETITIONDISTANCE < 500   THEN '<500m'
                WHEN COMPETITIONDISTANCE <= 2000 THEN '500m-2km'
                WHEN COMPETITIONDISTANCE <= 5000 THEN '2km-5km'
                ELSE '>5km'
            END AS competition_distance_tier,
            IS_PROMO2 AS is_promo2,
            PROMO2SINCEWEEK AS promo2_since_week,
            PROMO2SINCEYEAR AS promo2_since_year,
            PROMOINTERVAL AS promo_interval
        FROM RETAIL_CAPSTONE_DB.RAW.STORE_CLEANED
    """),

    ("GOLD.FACT_SALES", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.FACT_SALES AS
        SELECT
            MD5(CONCAT(CAST(STORE AS VARCHAR), '_', CAST(DATE AS VARCHAR))) AS sales_fact_key,
            STORE                                                           AS store_id,
            DATE                                                            AS sale_date,
            DAYOFWEEK                                                       AS day_of_week,
            SALES                                                           AS sales_amount,
            CUSTOMERS                                                       AS customer_count,
            ROUND(SALES / NULLIF(CUSTOMERS, 0), 2)                          AS sales_per_customer,
            PROMO                                                           AS is_promo,
            PROMO                                                           AS is_promo_active,
            STATEHOLIDAY                                                    AS state_holiday_code,
            CASE WHEN STATEHOLIDAY != '0' THEN 1 ELSE 0 END                 AS is_state_holiday,
            SCHOOLHOLIDAY                                                   AS is_school_holiday,
            CASE WHEN DAYOFWEEK IN (6, 7) THEN 1 ELSE 0 END                 AS is_weekend
        FROM RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN
        WHERE OPEN = 1 AND SALES > 0
    """),

    ("GOLD.FACT_STORE_CLOSURES", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.FACT_STORE_CLOSURES AS
        SELECT
            MD5(CONCAT(CAST(STORE AS VARCHAR), '_', CAST(DATE AS VARCHAR))) AS closure_fact_key,
            STORE                                                           AS store_id,
            DATE                                                            AS sale_date,
            DAYOFWEEK                                                       AS day_of_week,
            SALES                                                           AS sales_amount,
            CUSTOMERS                                                       AS customer_count,
            OPEN                                                            AS is_open,
            STATEHOLIDAY                                                    AS state_holiday_code,
            SCHOOLHOLIDAY                                                   AS is_school_holiday,
            CASE 
                WHEN STATEHOLIDAY IN ('a', 'b', 'c') THEN 'State Holiday Closure'
                WHEN SCHOOLHOLIDAY = 1               THEN 'School Holiday Closure'
                WHEN DAYOFWEEK = 7                   THEN 'Sunday / Weekend Closure'
                ELSE 'Operational / Refurbishment Closure'
            END AS closure_reason
        FROM RETAIL_CAPSTONE_DB.RAW.SALES_CLEAN
        WHERE OPEN = 0 OR SALES = 0
    """),

    ("GOLD.AGG_STORE_PERFORMANCE", """
        CREATE OR REPLACE TABLE RETAIL_CAPSTONE_DB.GOLD.AGG_STORE_PERFORMANCE AS
        SELECT
            s.store_id,
            st.store_type_name,
            st.assortment_name,
            COUNT(s.sale_date)                             AS total_open_days,
            ROUND(SUM(s.sales_amount), 2)                  AS total_revenue,
            ROUND(AVG(s.sales_amount), 2)                  AS avg_daily_sales,
            SUM(s.customer_count)                          AS total_customers,
            ROUND(AVG(s.customer_count), 1)                AS avg_daily_customers,
            ROUND(AVG(s.sales_per_customer), 2)            AS avg_basket_size,
            ROUND(AVG(CASE WHEN s.is_promo = 1 THEN s.sales_amount END), 2) AS avg_promo_sales,
            ROUND(AVG(CASE WHEN s.is_promo = 0 THEN s.sales_amount END), 2) AS avg_non_promo_sales,
            ROUND(
                (AVG(CASE WHEN s.is_promo = 1 THEN s.sales_amount END) - 
                 AVG(CASE WHEN s.is_promo = 0 THEN s.sales_amount END)) / 
                NULLIF(AVG(CASE WHEN s.is_promo = 0 THEN s.sales_amount END), 0) * 100, 2
            ) AS promo_lift_pct
        FROM RETAIL_CAPSTONE_DB.GOLD.FACT_SALES s
        JOIN RETAIL_CAPSTONE_DB.GOLD.DIM_STORE st ON s.store_id = st.store_id
        GROUP BY s.store_id, st.store_type_name, st.assortment_name
    """)
]

# 4. Execute all models in Snowflake automatically
for model_name, query_sql in gold_models:
    spark.read.format("snowflake").options(**sfOptions).option("query", query_sql).load()
    print(f"  ✅ Rebuilt {model_name}")

print("🎉 Success! All 5 Gold tables are 100% updated in Snowflake.")

# 5. Display verification audit table
audit_query = """
SELECT 'DIM_STORE' AS table_name, COUNT(*) AS row_count, NULL AS latest_date FROM RETAIL_CAPSTONE_DB.GOLD.DIM_STORE
UNION ALL
SELECT 'DIM_DATE', COUNT(*), MAX(DATE_ID) FROM RETAIL_CAPSTONE_DB.GOLD.DIM_DATE
UNION ALL
SELECT 'FACT_SALES', COUNT(*), MAX(SALE_DATE) FROM RETAIL_CAPSTONE_DB.GOLD.FACT_SALES
UNION ALL
SELECT 'FACT_STORE_CLOSURES', COUNT(*), MAX(SALE_DATE) FROM RETAIL_CAPSTONE_DB.GOLD.FACT_STORE_CLOSURES
UNION ALL
SELECT 'AGG_STORE_PERFORMANCE', COUNT(*), NULL FROM RETAIL_CAPSTONE_DB.GOLD.AGG_STORE_PERFORMANCE;
"""

display(spark.read.format("snowflake").options(**sfOptions).option("query", audit_query).load())
